- Uso CIMA para todo o ya es liarse
  
- Que hago cuando hay mas de un codigo ATC con el que comparar q tiene los 5 primeros diferentes

- Lo de los efectos adversos
  SOLO IMPRIMIR
- Deberia subir al git los notebooks de prueba?
  NO
- Cuando vuelvo a buscar opciones tengo que comparar cada una de las opciones con cada uno de los farmacos? Es decir: si solo quito uno? O todas las posibles opciones con todas las posibles opciones? que me puede llegar a parecer mas facil, pero entonces como diferencio que farmaco viene de que?
  SIN SOLUCION MUCHOS BUCLES Y MUCHOS IFELSE


Reducido a solo combinar dos ppio_act

In [2]:
import pandas as pd
DDI = pd.read_csv("DDI_sea.csv")
efectos = pd.read_csv("efectos_adversos.csv")

In [4]:
efectos.head()

,Drug_ATC,Drug_name,Side_effect,Freq_media
0,A16AA01,carnitine,Abdominal pain,0.116000
1,A16AA01,carnitine,Gastrointestinal pain,0.116000
2,A16AA01,carnitine,Amblyopia,0.036667
3,A16AA01,carnitine,Anaemia,0.058000
4,A16AA01,carnitine,Decreased appetite,0.042500


In [181]:
uno = DDI[DDI["Drug_name"] == "Clopidogrel"]
dos = DDI[DDI["Drug_name"] == "Omeprazole"]
uno
dos
#Hacer comprobacion primero de si no es nulo
p_uno = uno[uno["Tipo"]=="enzyme" ]["Gene_name"].tolist()
p_dos = dos[dos["Tipo"]=="enzyme" ]["Gene_name"].tolist()
lista= list(set(p_uno) & set(p_dos))

In [217]:
uno

,DrugBank_id,Drug_ATC,Drug_name,Tipo,Gene_name,Accion,Prioridad
8657,DB00758,B01AC04,Clopidogrel,target,P2RY12,antagonist,0.0
8658,DB00758,B01AC04,Clopidogrel,enzyme,CYP2C9,substrate|inhibitor,0.0
8659,DB00758,B01AC04,Clopidogrel,enzyme,CYP1A2,substrate,0.0
8660,DB00758,B01AC04,Clopidogrel,enzyme,CYP3A4,substrate,0.0
8661,DB00758,B01AC04,Clopidogrel,enzyme,CYP3A5,substrate,0.0
8662,DB00758,B01AC04,Clopidogrel,enzyme,CYP2C19,substrate,1.0


In [242]:
dos

,DrugBank_id,Drug_ATC,Drug_name,Tipo,Gene_name,Accion,Prioridad
3612,DB00338,A02BC51,Omeprazole,target,ATP4A,inhibitor,0.0
3613,DB00338,A02BC01,Omeprazole,target,ATP4A,inhibitor,0.0
3614,DB00338,A02BD05,Omeprazole,target,ATP4A,inhibitor,0.0
3615,DB00338,A02BD16,Omeprazole,target,ATP4A,inhibitor,0.0
3616,DB00338,A02BD01,Omeprazole,target,ATP4A,inhibitor,0.0
3617,DB00338,A02BC51,Omeprazole,target,ATP4B,modulator,0.0
3618,DB00338,A02BC01,Omeprazole,target,ATP4B,modulator,0.0
3619,DB00338,A02BD05,Omeprazole,target,ATP4B,modulator,0.0
3620,DB00338,A02BD16,Omeprazole,target,ATP4B,modulator,0.0
3621,DB00338,A02BD01,Omeprazole,target,ATP4B,modulator,0.0


In [135]:
def comparacion (df_1,df_2):
    enzima_1 = df_1[df_1["Tipo"]=="enzyme" ]["Gene_name"].tolist()
    enzima_2 = df_2[df_2["Tipo"]=="enzyme" ]["Gene_name"].tolist()
    coincidentes = list(set(enzima_1) & set(enzima_2))
    return coincidentes


#PROBABLEMENTE HACER TEXTOS DE TODOS LOS POSIBLES RESULTADOS Y QUE LOS SAQUE Y PUNTO

In [203]:
def calcular_riesgo (coincidentes, ppio_act, comparo, principal, segundo):
    #RIESGOS ALTOS
    #Como puede haber mas de uno principal
    if len(principal)>1 and len(segundo)>1:
        if list(set(principal) & set(segundo)):
            print(f"Riesgo de interaccion entre {ppio_act} y {comparo} ALTO")
            
    #Principal es lista
    elif len(principal)>1:
        if segundo[0] in principal:
            print(f"Riesgo de interaccion entre {ppio_act} y {comparo} ALTO")
            
    elif len(segundo)>1:
        if principal[0] in segundo:
            print(f"Riesgo de interaccion entre {ppio_act} y {comparo} ALTO")
            
    
    #Si es la de los dos ALTO (dos principales) estara en coincidentes si llega hasta aqui
    elif principal[0] == segundo[0] :
        print(f"Riesgo de interaccion entre {ppio_act} y {comparo} ALTO")          #Describirlo

    
    #RIESGO MEDIO
    #Si es solo la de uno MEDIO
    else:
        print(f"Riesgo de interacción entre {ppio_act} y {comparo} MEDIO")       #Describirlo

In [240]:
def busco_opciones(DDI,ppio_ini, ATC_ini, ppio_comp, ATC_comp):
    opciones = {}
    ATC = [ATC_ini, ATC_comp]
    for i in range(len(ATC)):
        codigo_referencia = ATC[i][:5]
        resultado = DDI[DDI['Drug_ATC'].str.startswith(codigo_referencia, na=False)]["Drug_name"].unique().tolist()
        if i == 0:
            opciones[ppio_ini] = resultado
        else:
            opciones[ppio_comp] = resultado

In [205]:
prescripcion = ["Omeprazole","Clopidogrel"]

In [207]:
for i in range(len(prescripcion)):
    for j in range(i + 1, len(prescripcion)):

        ppio_act = prescripcion[i]
        comparo = prescripcion[j]
        
        #Me va a dar error si no es talcual asiq split().lower() deberia
        inicial = DDI[DDI["Drug_name"] == ppio_act]
        secundario = DDI[DDI["Drug_name"] == comparo]
        
        #Comparo ambos
        coincidentes = comparacion(inicial, secundario)

        #Vemos las vias principales
        principal = inicial[inicial["Prioridad"]==1.0]["Gene_name"].tolist()
        segundo = secundario[secundario["Prioridad"]==1.0]["Gene_name"].tolist()
        #Cuando calculamos el riesgo no devolvemos nada, solo texto
        if coincidentes:
            calcular_riesgo(coincidentes, ppio_act, comparo, principal, segundo)
            #En esta linea busco las opciones
            opciones = busco_opciones(DDI,ppio_ini, ATC_ini, ppio_comp, ATC_comp)

        
        else:
            #Cuando no coinciden es interaccion BAJA o leve
            print(f"Riesgo de interacción entre {ppio_act} y {comparo} BAJO")

llego
Riesgo de interaccion entre Omeprazole y Clopidogrel ALTO
